This notebook holds the conversion analysis. The standard population is US dealer leads, meaning non spam, not for sale by owner, on the five United States portals, over twelve months. A lead is counted as converted when the boat it is about appears in sold boats with a sale date on or after the lead day. Because a conversion is a one or zero flag, the average of that flag is the conversion rate.

This is conversion by listing price band. The empty band row is the overall baseline.

In [ ]:
WITH lds AS (
  SELECT LISTING_ID, DAY, PRODUCT_PRICE
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
    AND NOT IS_FSBO
    AND PORTAL_NAME IN ('boats.com','boat trader','bt app','yachtworld','yw app')
),
sold AS (SELECT LISTING_ID, MIN(DATE_SOLD) AS first_sold FROM AWS_DP_PROD.LISTINGS.SOLD_BOATS GROUP BY 1),
flagged AS (
  SELECT lds.*, IFF(sold.first_sold IS NOT NULL AND sold.first_sold >= lds.DAY,1,0) AS converted
  FROM lds LEFT JOIN sold ON sold.LISTING_ID = lds.LISTING_ID
)
SELECT
  CASE WHEN PRODUCT_PRICE < 25000  THEN 'a under 25k'
       WHEN PRODUCT_PRICE < 75000  THEN 'b 25 to 75k'
       WHEN PRODUCT_PRICE < 200000 THEN 'c 75 to 200k'
       WHEN PRODUCT_PRICE < 500000 THEN 'd 200 to 500k'
       ELSE 'e 500k plus' END      AS price_band,
  COUNT(*)                     AS leads_in_bucket,
  SUM(converted)               AS boats_that_later_sold,
  ROUND(AVG(converted)*100, 1) AS pct_sold
FROM flagged
WHERE PRODUCT_PRICE IS NOT NULL
GROUP BY GROUPING SETS ((price_band), ())
ORDER BY price_band NULLS LAST;

This is conversion across four buyer and lead level signals, new versus returning visitor, message length, buyer to boat distance, and whether a phone number was included. 

In [ ]:
WITH lds AS (
  SELECT LISTING_ID, DAY, NEW_VS_RETURNED, CUSTOMER_COMMENTS,
         USER_LISTING_DISTANCE_IN_MILES, PHONE_NUMBER
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
    AND NOT IS_FSBO
    AND PORTAL_NAME IN ('boats.com','boat trader','bt app','yachtworld','yw app')
),
sold AS (SELECT LISTING_ID, MIN(DATE_SOLD) AS fs FROM AWS_DP_PROD.LISTINGS.SOLD_BOATS GROUP BY 1),
f AS (SELECT lds.*, IFF(sold.fs IS NOT NULL AND sold.fs >= lds.DAY,1,0) AS conv
      FROM lds LEFT JOIN sold ON sold.LISTING_ID = lds.LISTING_ID)
SELECT 'new_vs_returning' AS signal, LOWER(NEW_VS_RETURNED) AS bucket, COUNT(*) AS leads, ROUND(AVG(conv)*100,1) AS pct_sold
  FROM f WHERE NEW_VS_RETURNED IS NOT NULL GROUP BY 2
UNION ALL
SELECT 'message_length',
  CASE WHEN NULLIF(TRIM(CUSTOMER_COMMENTS),'') IS NULL THEN 'a none'
       WHEN LENGTH(CUSTOMER_COMMENTS)<=100 THEN 'b short 1 to 100'
       WHEN LENGTH(CUSTOMER_COMMENTS)<=300 THEN 'c medium 101 to 300' ELSE 'd long 300 plus' END,
  COUNT(*), ROUND(AVG(conv)*100,1) FROM f GROUP BY 2
UNION ALL
SELECT 'distance',
  CASE WHEN USER_LISTING_DISTANCE_IN_MILES<25 THEN 'a under 25mi'
       WHEN USER_LISTING_DISTANCE_IN_MILES<100 THEN 'b 25 to 100'
       WHEN USER_LISTING_DISTANCE_IN_MILES<500 THEN 'c 100 to 500' ELSE 'd 500 plus' END,
  COUNT(*), ROUND(AVG(conv)*100,1) FROM f WHERE USER_LISTING_DISTANCE_IN_MILES IS NOT NULL GROUP BY 2
UNION ALL
SELECT 'phone_in_lead', IFF(NULLIF(TRIM(PHONE_NUMBER),'') IS NOT NULL,'yes','no'),
  COUNT(*), ROUND(AVG(conv)*100,1) FROM f GROUP BY 2
ORDER BY 1,2;

This shows that the number of leads a listing receives strongly predicts whether it sells. Each row groups listings by how many US dealer leads they received and reports the share that later sold.

In [ ]:
WITH lds AS (
  SELECT LISTING_ID, DAY
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
    AND NOT IS_FSBO
    AND PORTAL_NAME IN ('boats.com','boat trader','bt app','yachtworld','yw app')
),
per_listing AS (SELECT LISTING_ID, COUNT(*) AS n_leads, MIN(DAY) AS first_lead FROM lds GROUP BY 1),
sold AS (SELECT LISTING_ID, MIN(DATE_SOLD) AS fs FROM AWS_DP_PROD.LISTINGS.SOLD_BOATS GROUP BY 1),
j AS (
  SELECT p.n_leads, IFF(sold.fs IS NOT NULL AND sold.fs >= p.first_lead,1,0) AS sold_flag
  FROM per_listing p LEFT JOIN sold ON sold.LISTING_ID = p.LISTING_ID
)
SELECT
  CASE WHEN n_leads=1 THEN 'a 1 lead'
       WHEN n_leads BETWEEN 2 AND 3 THEN 'b 2 to 3'
       WHEN n_leads BETWEEN 4 AND 9 THEN 'c 4 to 9' ELSE 'd 10 plus' END AS lead_count_bucket,
  COUNT(*) AS listings, ROUND(AVG(sold_flag)*100,1) AS pct_sold
FROM j GROUP BY 1 ORDER BY 1;

This is the observable outcome by segment. It shows that dealers have a sale outcome but no inbox engagement, while for sale by owner has inbox engagement but no sale record.


In [ ]:
WITH base AS (
  SELECT
    CASE WHEN IS_FSBO THEN 'FSBO'
         WHEN PORTAL_NAME IN ('boats.com','boat trader','bt app','yachtworld','yw app') THEN 'US dealer'
         ELSE 'Intl or other dealer' END AS segment,
    LISTING_ID, DAY,
    IFF(LEAD_FOUND_IN_INBOX_CONVERSATIONS = TRUE,1,0)                     AS in_inbox,
    IFF(COALESCE(INBOX_CONVERSATION_NUM_MESSAGES_FROM_SELLER,0) >= 1,1,0) AS seller_msg,
    IFF(COALESCE(INBOX_CONVERSATION_NUM_MESSAGES_FROM_BUYER,0)  >= 2,1,0) AS buyer_2plus
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY >= DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
),
sold AS (SELECT LISTING_ID, MIN(DATE_SOLD) AS fs FROM AWS_DP_PROD.LISTINGS.SOLD_BOATS GROUP BY 1)
SELECT
  segment, COUNT(*) AS leads,
  ROUND(AVG(IFF(s.fs IS NOT NULL AND s.fs >= base.DAY,1,0))*100,1) AS pct_listing_sold_after_lead,
  ROUND(AVG(in_inbox)*100,1)    AS pct_found_in_inbox,
  ROUND(AVG(seller_msg)*100,1)  AS pct_seller_sent_msg,
  ROUND(AVG(buyer_2plus)*100,1) AS pct_buyer_2plus_msgs
FROM base LEFT JOIN sold s ON s.LISTING_ID = base.LISTING_ID
GROUP BY 1 ORDER BY leads DESC;

This is the self reported intent from the authentication questions, joined to leads by buyer email and portal, where the answer was given on or before the lead. The first result shows coverage, and the second tests whether owning a boat or a near term buying plan predicts the sale outcome.

In [ ]:
WITH lds AS (
  SELECT LOWER(SENDER_EMAIL) AS em, PORTAL_NAME AS pn, CREATED_TIME AS ct, LISTING_ID, DAY
  FROM AWS_DP_PROD.CONNECTIONS.EMAIL_LEADS
  WHERE DAY>=DATEADD(month,-12,CURRENT_DATE) AND COALESCE(IS_SPAM_FINAL,0)=0
    AND NOT IS_FSBO AND PORTAL_NAME IN ('boats.com','boat trader','bt app','yachtworld','yw app')
),
q AS (
  SELECT LOWER(username) AS em, portal_name AS pn,
    MAX(CASE WHEN key_question='do-you-currently-own-a-boat' AND answer<>'' THEN answer END) AS own_boat,
    MAX(CASE WHEN key_question='when-do-you-plan-to-buy' AND answer<>'' THEN answer END) AS timeframe,
    MIN(question_answered_date) AS fa
  FROM AWS_DP_PROD."USER".USER_AUTHENTICATION_QNA GROUP BY 1,2
),
sold AS (SELECT LISTING_ID, MIN(DATE_SOLD) AS fs FROM AWS_DP_PROD.LISTINGS.SOLD_BOATS GROUP BY 1),
f AS (
  SELECT IFF(sold.fs IS NOT NULL AND sold.fs>=lds.DAY,1,0) AS conv, q.own_boat
  FROM lds
  LEFT JOIN q ON q.em=lds.em AND q.pn=lds.pn AND q.fa<=lds.ct
  LEFT JOIN sold ON sold.LISTING_ID=lds.LISTING_ID
)
SELECT COALESCE(own_boat,'no answer') AS owns_a_boat, COUNT(*) AS leads, ROUND(AVG(conv)*100,1) AS pct_sold
FROM f GROUP BY 1 ORDER BY leads DESC;